In [0]:
# ============================================================

# SILVER – EasyVista demo with realistic re-ingestion

# ============================================================

from pyspark.sql import functions as F

from pyspark.sql.types import *

from datetime import datetime, timedelta

# ---------------- CONFIG ----------------

DB = "workspace.slainte_silver"

TABLE = f"{DB}.easyvista_tickets_silver_demo"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB}")

now = datetime(2026, 1, 10, 10, 0, 0)

rows = [

    # ===============================

    # uploader_1 – ProjAlpha (EasyVista)

    # Same ticket ingested twice

    # ===============================

    (

        "IT", "INC20260001", "Incident", 1,

        "L1 Support", None, "Open",

        datetime(2026,1,1,9,0,0),

        None,

        datetime(2026,1,1,9,0,0),

        "easyvista", "uploader_1", "ProjAlpha",

        now

    ),

    (

        "IT", "INC20260001", "Incident", 1,

        "L1 Support", "Alice Alpha", "In Progress",

        datetime(2026,1,1,9,0,0),

        datetime(2026,1,1,9,20,0),

        datetime(2026,1,1,10,30,0),

        "easyvista", "uploader_1", "ProjAlpha",

        now + timedelta(hours=1)

    ),

    # ===============================

    # uploader_1 – ProjBeta (ServiceNow)

    # ===============================

    (

        "IT", "REQ20260002", "Service", 3,

        "Apps", "Bob Beta", "Resolved",

        datetime(2026,1,2,11,0,0),

        datetime(2026,1,2,11,10,0),

        datetime(2026,1,2,15,45,0),

        "servicenow", "uploader_1", "ProjBeta",

        now

    ),

    # ===============================

    # uploader_2 – ProjGamma (Jira)

    # Ticket updated 3 times

    # ===============================

    (

        "IT", "INC20260003", "Incident", 2,

        "Network", None, "Open",

        datetime(2026,1,3,8,30,0),

        None,

        datetime(2026,1,3,8,30,0),

        "jira", "uploader_2", "ProjGamma",

        now

    ),

    (

        "IT", "INC20260003", "Incident", 2,

        "Network", "Charlie Gamma", "In Progress",

        datetime(2026,1,3,8,30,0),

        datetime(2026,1,3,8,45,0),

        datetime(2026,1,3,10,0,0),

        "jira", "uploader_2", "ProjGamma",

        now + timedelta(hours=2)

    ),

    (

        "IT", "INC20260003", "Incident", 2,

        "Network", "Charlie Gamma", "Resolved",

        datetime(2026,1,3,8,30,0),

        datetime(2026,1,3,8,45,0),

        datetime(2026,1,3,13,20,0),

        "jira", "uploader_2", "ProjGamma",

        now + timedelta(hours=5)

    ),

    # ===============================

    # uploader_3 – ProjDelta (Zabbix)

    # ===============================

    (

        "IT", "PRB20260004", "Problem", 4,

        "L2 Support", "Diana Delta", "Open",

        datetime(2026,1,4,14,0,0),

        datetime(2026,1,4,14,30,0),

        datetime(2026,1,4,14,30,0),

        "zabbix", "uploader_3", "ProjDelta",

        now

    )

]

schema = StructType([

    StructField("entity", StringType()),

    StructField("ticket_number", StringType()),

    StructField("ticket_type", StringType()),

    StructField("priority", IntegerType()),

    StructField("support_group", StringType()),

    StructField("assignee", StringType()),

    StructField("status", StringType()),

    StructField("created_at", TimestampType()),

    StructField("assigned_at", TimestampType()),

    StructField("last_update", TimestampType()),

    StructField("source", StringType()),

    StructField("source_user_id", StringType()),

    StructField("project", StringType()),

    StructField("ingestion_ts", TimestampType()),

])

df_silver = spark.createDataFrame(rows, schema)

df_silver.write.format("delta").mode("overwrite").saveAsTable(TABLE)

print("✅ SILVER demo written with realistic re-ingestion")

display(spark.table(TABLE).orderBy("ticket_number", "ingestion_ts"))
 

In [0]:
# ============================================================

# SILVER – SLA demo table (client-defined SLA, normalized)

# ============================================================

from pyspark.sql import functions as F

from pyspark.sql.types import *

# ---------- CONFIG ----------

DB = "workspace.slainte_silver"

TABLE = f"{DB}.sla_silver_demo"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB}")

# ------------------------------------------------------------

# Demo SLA data

#

# - uploader_1 uses TEXT priorities

# - uploader_2 uses NUMERIC priorities

# - uploader_3 has project-specific SLA

# ------------------------------------------------------------

sla_rows = [

    # uploader_1 – ProjAlpha (TEXT priorities)

    ("uploader_1", "ProjAlpha", "High",   1, 15, 4),

    ("uploader_1", "ProjAlpha", "Medium", 2, 30, 8),

    ("uploader_1", "ProjAlpha", "Low",    3, 60, 24),

    # uploader_1 – ProjBeta (different SLA)

    ("uploader_1", "ProjBeta", "High",   1, 20, 6),

    ("uploader_1", "ProjBeta", "Medium", 2, 45, 12),

    ("uploader_1", "ProjBeta", "Low",    3, 90, 36),

    # uploader_2 – ProjGamma (NUMERIC priorities)

    ("uploader_2", "ProjGamma", "1", 1, 15, 4),

    ("uploader_2", "ProjGamma", "2", 2, 30, 8),

    ("uploader_2", "ProjGamma", "3", 3, 60, 24),

    ("uploader_2", "ProjGamma", "4", 4, 120, 48),

    ("uploader_2", "ProjGamma", "5", 5, 240, 72),

    # uploader_3 – ProjDelta (FIXED: added P4)

    ("uploader_3", "ProjDelta", "P1", 1, 10, 2),

    ("uploader_3", "ProjDelta", "P2", 2, 30, 6),

    ("uploader_3", "ProjDelta", "P3", 3, 90, 24),

    ("uploader_3", "ProjDelta", "P4", 4, 180, 48),  # ✅ NEW

]

# ------------------------------------------------------------

# Schema

# ------------------------------------------------------------

schema = StructType([

    StructField("source_user_id", StringType()),

    StructField("project", StringType()),

    StructField("priority_raw", StringType()),       # client-defined

    StructField("priority_level", IntegerType()),    # normalized

    StructField("response_time_minutes", IntegerType()),

    StructField("resolution_time_hours", IntegerType()),

])

df_sla = spark.createDataFrame(sla_rows, schema)

# ------------------------------------------------------------

# Write SILVER SLA table

# ------------------------------------------------------------

df_sla.write.format("delta").mode("overwrite").saveAsTable(TABLE)

print(f"✅ SILVER SLA demo table written: {TABLE}")

display(

    spark.table(TABLE)

         .orderBy("source_user_id", "project", "priority_level")

)
 

In [0]:
# SILVER: penalty_silver_demo

from pyspark.sql import functions as F

from pyspark.sql.types import *

from datetime import datetime, timedelta

DB = "workspace.slainte_silver"

TABLE = f"{DB}.penalty_silver_demo"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB}")

# Demo rows:

# fields:

# source_user_id, project, penalty_code, priority_raw, priority_level (nullable),

# penalty_type, penalty_value, penalty_unit, conditions (json string), effective_from, effective_to, created_by, created_at

now = datetime.utcnow()

rows = [

    # uploader_1 / ProjAlpha: text priorities

    ("uploader_1", "ProjAlpha", "LATE_RESPONSE", "High", 1, "fixed", 50.0, "USD", '{"event":"response_breach"}', now, None, "admin", now),

    ("uploader_1", "ProjAlpha", "LATE_RESOLUTION", "High", 1, "fixed", 150.0, "USD", '{"event":"resolution_breach"}', now, None, "admin", now),

    ("uploader_1", "ProjAlpha", "LATE_RESPONSE", "Medium", 2, "fixed", 30.0, "USD", '{"event":"response_breach"}', now, None, "admin", now),

    # uploader_1 / ProjBeta: different SLA/penalty values

    ("uploader_1", "ProjBeta", "LATE_RESPONSE", "High", 1, "percent", 10.0, "%", '{"event":"response_breach"}', now, None, "admin", now),

    # uploader_2 / ProjGamma: numeric priorities

    ("uploader_2", "ProjGamma", "LATE_RESPONSE", "1", 1, "fixed", 25.0, "USD", '{"event":"response_breach"}', now, None, "admin", now),

    ("uploader_2", "ProjGamma", "LATE_RESOLUTION", "1", 1, "fixed", 100.0, "USD", '{"event":"resolution_breach"}', now, None, "admin", now),

    # uploader_3 / ProjDelta: small sample

    ("uploader_3", "ProjDelta", "SERVICE_PENALTY", "P1", 1, "fixed", 75.0, "USD", '{"event":"any_breach"}', now, None, "admin", now),

]

schema = StructType([

    StructField("source_user_id", StringType()),

    StructField("project", StringType()),

    StructField("penalty_code", StringType()),

    StructField("priority_raw", StringType()),

    StructField("priority_level", IntegerType()),     # normalized priority level (if known)

    StructField("penalty_type", StringType()),        # "fixed" | "percent" | ...

    StructField("penalty_value", DoubleType()),       # numeric

    StructField("penalty_unit", StringType()),        # "USD", "%", "points"

    StructField("conditions", StringType()),          # free-form JSON/text describing filters

    StructField("effective_from", TimestampType()),

    StructField("effective_to", TimestampType()),

    StructField("created_by", StringType()),

    StructField("created_at", TimestampType()),

])

df_pen_silver = spark.createDataFrame(rows, schema)

# Normalize: ensure priority_level present (try to parse from priority_raw if priority_level is null)

df_pen_silver = df_pen_silver.withColumn(

    "priority_level",

    F.coalesce(

        F.col("priority_level"),

        F.regexp_extract(F.col("priority_raw"), r"(\d+)", 1).cast("int")

    )

)

# Save to SILVER

df_pen_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE)

print(f"✅ Wrote SILVER penalty demo: {TABLE} (rows: {df_pen_silver.count()})")

display(spark.table(TABLE).orderBy("source_user_id", "project", "penalty_code"))
 

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.slainte_silver.v_ticket_sla_final AS

WITH latest_ticket AS (

    SELECT

        t.*,

        ROW_NUMBER() OVER (

            PARTITION BY

                t.source_user_id,

                t.project,

                t.ticket_number

            ORDER BY

                t.ingestion_ts DESC

        ) AS rn

    FROM workspace.slainte_silver.easyvista_tickets_silver_demo t

),

ticket_current AS (

    SELECT *

    FROM latest_ticket

    WHERE rn = 1

),

ticket_with_sla AS (

    SELECT

        tc.source_user_id,

        tc.project,

        tc.ticket_number,

        tc.created_at,

        tc.assigned_at,

        tc.last_update,

        tc.priority                     AS priority_code,

        s.response_time_minutes,

        s.resolution_time_hours,

        -- =========================

        -- RESPONSE SLA

        -- =========================

        CASE

            WHEN tc.assigned_at IS NOT NULL

             AND s.response_time_minutes IS NOT NULL

             AND (

                 (unix_timestamp(tc.last_update) - unix_timestamp(tc.assigned_at)) / 60

             ) > s.response_time_minutes

            THEN 1

            ELSE 0

        END AS response_breach_flag,

        -- =========================

        -- RESOLUTION SLA

        -- =========================

        CASE

            WHEN s.resolution_time_hours IS NOT NULL

             AND (

                 (unix_timestamp(tc.last_update) - unix_timestamp(tc.created_at)) / 3600

             ) > s.resolution_time_hours

            THEN 1

            ELSE 0

        END AS resolution_breach_flag,

        -- =========================

        -- ELAPSED TIMES

        -- =========================

        (unix_timestamp(tc.last_update) - unix_timestamp(tc.assigned_at)) / 60

            AS response_minutes_elapsed,

        (unix_timestamp(tc.last_update) - unix_timestamp(tc.created_at)) / 3600

            AS resolution_hours_elapsed

    FROM ticket_current tc

    LEFT JOIN workspace.slainte_silver.sla_silver_demo s

        ON  tc.source_user_id = s.source_user_id

        AND tc.project        = s.project

        AND tc.priority       = s.priority_level

)

SELECT

    source_user_id,

    project,

    ticket_number,

    created_at,

    assigned_at,

    last_update,

    priority_code,

    response_time_minutes,

    resolution_time_hours,

    response_minutes_elapsed,

    resolution_hours_elapsed,

    response_breach_flag,

    resolution_breach_flag,

    CASE

        WHEN response_breach_flag = 1

          OR resolution_breach_flag = 1

        THEN 1

        ELSE 0

    END AS sla_breached_flag

FROM ticket_with_sla;
 

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.slainte_silver.v_ticket_penalty_final AS

WITH latest_ticket AS (

    SELECT

        t.*,

        ROW_NUMBER() OVER (

            PARTITION BY

                t.source_user_id,

                t.project,

                t.ticket_number

            ORDER BY

                t.ingestion_ts DESC

        ) AS rn

    FROM workspace.slainte_silver.easyvista_tickets_silver_demo t

),

ticket_current AS (

    SELECT *

    FROM latest_ticket

    WHERE rn = 1

),

ticket_sla AS (

    SELECT *

    FROM workspace.slainte_silver.v_ticket_sla_final

),

penalty_applied AS (

    SELECT

        tc.source_user_id,

        tc.project,

        tc.ticket_number,

        tc.priority                   AS priority_level,

        -- SLA flags

        sla.response_breach_flag,

        sla.resolution_breach_flag,

        -- Penalty rule

        p.penalty_code,

        p.penalty_type,

        p.penalty_value,

        p.penalty_unit,

        -- Determine if penalty applies

        CASE

            WHEN sla.response_breach_flag = 1

             AND p.penalty_code = 'LATE_RESPONSE'

            THEN 1

            WHEN sla.resolution_breach_flag = 1

             AND p.penalty_code = 'LATE_RESOLUTION'

            THEN 1

            ELSE 0

        END AS penalty_applies_flag

    FROM ticket_current tc

    LEFT JOIN ticket_sla sla

        ON  tc.source_user_id = sla.source_user_id

        AND tc.project        = sla.project

        AND tc.ticket_number = sla.ticket_number

    LEFT JOIN workspace.slainte_silver.penalty_silver_demo p

        ON  tc.source_user_id = p.source_user_id

        AND tc.project        = p.project

        AND tc.priority       = p.priority_level

),

penalty_final AS (

    SELECT

        source_user_id,

        project,

        ticket_number,

        priority_level,

        MAX(response_breach_flag)   AS response_breach_flag,

        MAX(resolution_breach_flag) AS resolution_breach_flag,

        -- Penalty info (only if applied)

        MAX(

            CASE WHEN penalty_applies_flag = 1 THEN penalty_code END

        ) AS penalty_code,

        MAX(

            CASE WHEN penalty_applies_flag = 1 THEN penalty_type END

        ) AS penalty_type,

        MAX(

            CASE WHEN penalty_applies_flag = 1 THEN penalty_value END

        ) AS penalty_value,

        MAX(

            CASE WHEN penalty_applies_flag = 1 THEN penalty_unit END

        ) AS penalty_unit,

        -- Final penalty amount

        MAX(

            CASE

                WHEN penalty_applies_flag = 1 THEN penalty_value

                ELSE 0

            END

        ) AS penalty_amount

    FROM penalty_applied

    GROUP BY

        source_user_id,

        project,

        ticket_number,

        priority_level

)

SELECT *

FROM penalty_final;
 